# 第 3 课：大型语言模型能掌握 Wordle 吗？

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>访问 <code>requirements.txt</code> 文件：</b> 1) 点击 notebook 顶部菜单中的 <em>"File"</em>，然后 2) 点击 <em>"Open"</em>。

<p> ⬇ &nbsp; <b>下载 Notebook：</b> 1) 点击 notebook 顶部菜单中的 <em>"File"</em>，然后 2) 点击 <em>"Download as"</em> 并选择 <em>"Notebook (.ipynb)"</em>。</p>

<p> 📒 &nbsp; 如需更多帮助，请参阅 <em>"Appendix – Tips, Help, and Download"</em> 课程。</p>

</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>运行结果可能不同：</b> AI 聊天模型生成的输出具有动态性和概率性，因此每次执行的结果都可能不同。如果你的结果和视频中的展示不一致，不必惊讶。</p>

先加载依赖并设置 Predibase 客户端，后续你会用它来调用基础模型和微调后的模型：

In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

# 初始化客户端，以便通过 OpenAI 客户端在 Predibase 上查询 Qwen2.5 7B Instruct。

_ = load_dotenv()

client = OpenAI(
    base_url=os.environ["PREDIBASE_MODEL_QWEN_URL"], # Qwen 2.5 7B Instruct 模型
    api_key=os.environ["PREDIBASE_API_KEY"],
)

从 Hugging Face 加载模型分词器：

In [2]:
from transformers import AutoTokenizer

base_model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 设置用于玩 Wordle 的提示词

In [3]:
SYSTEM_PROMPT = """
You are playing Wordle, a word-guessing game.

### Game Rules:
- You have **6 tries** to guess a secret **5-letter** word.
- Each guess must be a valid **5-letter English word**.
- After each guess, you will receive feedback indicating how close 
your guess was.

### Feedback Format:
Each letter in your guess will receive one of three symbols:
1. ✓ : The letter is in the word and in the CORRECT position.
2. - : The letter is in the word but in the WRONG position.
3. x : The letter is NOT in the word.

### Example:
Secret Word: BRISK

Guess 1: STORM → Feedback: S(-) T(x) O(x) R(-) M(x)
Guess 2: BRAVE → Feedback: B(✓) R(✓) A(x) V(x) E(x)
Guess 3: BRISK → Feedback: B(✓) R(✓) I(✓) S(✓) K(✓)

### Response Format:
Think through the problem and feedback step by step. Make sure to 
first add your step by step thought process within <think> </think> 
tags. Then, return your guessed word in the following format: 
<guess> guessed-word </guess>.
"""


In [4]:
from dataclasses import dataclass
from enum import Enum
from typing import List


class LetterFeedback(Enum):
    CORRECT = "✓"
    WRONG_POS = "-"
    WRONG_LETTER = "x"


@dataclass
class GuessWithFeedback:
    guess: str
    feedback: List[LetterFeedback]

    def __repr__(self) -> str:
        """Returns a readable string showing the guess alongside
        its letter-by-letter feedback."""
        feedback_str = " ".join(f"{letter}({fb.value})" for letter, fb in zip(self.guess, self.feedback))
        return f"{self.guess} → Feedback: {feedback_str}"

In [5]:
def render_user_prompt(past_guesses: List[GuessWithFeedback]) -> str:
    """Creates a user-facing prompt that includes past guesses 
    and their feedback."""
    prompt = "Make a new 5-letter word guess."
    if past_guesses:
        prompt += "\n\nHere is some previous feedback:"
        for i, guess in enumerate(past_guesses):
            prompt += f"\nGuess {i+1}: {guess}"
    return prompt

In [6]:
def render_prompt(past_guesses: List[GuessWithFeedback]):
    """Formats a full chat prompt using a system message, user 
    prompt, and assistant preamble to start the model's 
    step-by-step reasoning."""
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": render_user_prompt(past_guesses)
        },
        {
            "role": "assistant",
            "content": "Let me solve this step by step.\n<think>"
        }
    ]

    return tokenizer.apply_chat_template(
        messages, tokenize=False, continue_final_message=True
    )

In [ ]:
def generate_stream(prompt: str, adapter_id: str = "") -> str:
    """Streams a model-generated response from a prompt in 
    real-time and prints it as it arrives."""
    response = client.completions.create(
        model=adapter_id,
        prompt=prompt,
        # 为了评估结果，生成确定性的响应
        temperature=0.0, 
        max_tokens=2048,
        stream=True,
    )
    
    completion = ""
    for chunk in response:
        if chunk.choices[0].text is not None:
            content = chunk.choices[0].text
            print(content, end="", flush=True)
            completion += content
    print()

    return completion

## 进行一轮 Wordle

先构造提示词，并加入前两次猜测得到的反馈：

In [8]:
secret_word = "CRAFT"

past_guesses = [
    GuessWithFeedback(
        "CRANE", [
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.WRONG_LETTER, 
            LetterFeedback.WRONG_LETTER,
        ]),
    GuessWithFeedback(
        "CRASH", [
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.WRONG_LETTER, 
            LetterFeedback.WRONG_LETTER,
        ]),
]

prompt = render_prompt(past_guesses)
print(prompt)

<|im_start|>system

You are playing Wordle, a word-guessing game.

### Game Rules:
- You have **6 tries** to guess a secret **5-letter** word.
- Each guess must be a valid **5-letter English word**.
- After each guess, you will receive feedback indicating how close 
your guess was.

### Feedback Format:
Each letter in your guess will receive one of three symbols:
1. ✓ : The letter is in the word and in the CORRECT position.
2. - : The letter is in the word but in the WRONG position.
3. x : The letter is NOT in the word.

### Example:
Secret Word: BRISK

Guess 1: STORM → Feedback: S(-) T(x) O(x) R(-) M(x)
Guess 2: BRAVE → Feedback: B(✓) R(✓) A(x) V(x) E(x)
Guess 3: BRISK → Feedback: B(✓) R(✓) I(✓) S(✓) K(✓)

### Response Format:
Think through the problem and feedback step by step. Make sure to 
first add your step by step thought process within <think> </think> 
tags. Then, return your guessed word in the following format: 
<guess> guessed-word </guess>.
<|im_end|>
<|im_start|>user
Make

向基础模型发送提示词，并查看它的响应：

In [9]:
base_completion = generate_stream(prompt)

From the feedback, we know:
- C is in the correct position (1st letter)
- R is in the correct position (2nd letter)
- A is in the correct position (3rd letter)
- N is not in the word
- E is not in the word
- S is not in the word
- H is not in the word

This leaves us with the letters B, D, G, I, J, K, L, M, O, P, Q, T, U, V, W, X, Y, Z to choose from for the 4th and 5th positions. We should try to include letters that are common in 5-letter words and haven't been ruled out yet.</think>
<guess>CRANE</guess>


现在向 RFT 模型发送提示词，看看它如何响应：

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> <b>注意：</b> 下方单元会加载一个使用 RFT 训练、能够玩 Wordle 的模型。你会在后续课程中了解这个模型的训练方式，并在 L8 中看到其在 Predibase 平台上的完整训练配置。
</div>

In [ ]:
# 调用微调后的模型
ft_completion = generate_stream(prompt, adapter_id="wordle-dlai/2")

Let's analyze the feedback we have so far:

Guess 1: CRANE → Feedback: C(✓) R(✓) A(✓) N(x) E(x)
This means:
- C is in the correct position (1st position)
- R is in the correct position (2nd position)
- A is in the correct position (3rd position)
- N is not in the word
- E is not in the word

Guess 2: CRASH → Feedback: C(✓) R(✓) A(✓) S(x) H(x)
This confirms:
- C, R, A are in their correct positions (1st, 2nd, 3rd)
- S is not in the word
- H is not in the word

So we know:
- The first three letters are "CRA"
- The letters N, E, S, H are not in the word

Now I need to make a guess for the 4th and 5th positions. Let me think of 5-letter words that start with "CRA" and don't contain N, E, S, H.

Possible words:
- CRAFT
- CRAMP
- CRANK (no, contains N)
- CRAP
- CRAZE (no, contains E)
- CRAZY (no, contains Z)
- CRAB
- CRACK
- CRAM
- CRAYON (no, contains N)

Let's eliminate those with letters we know are not in the word.

From the remaining options, CRAFT, CRAMP, CRAP, CRAB, CRACK, and CRAM se

## 玩一整局 Wordle

先设置一个辅助函数，用来根据一次猜测返回反馈：

In [11]:
import re

def get_feedback(guess: str, secret_word: str) -> List[LetterFeedback]:
    valid_letters = set(secret_word)
    feedback = []
    for letter, secret_letter in zip(guess, secret_word):
        if letter == secret_letter:
            feedback.append(LetterFeedback.CORRECT)
        elif letter in valid_letters:
            feedback.append(LetterFeedback.WRONG_POS)
        else:
            feedback.append(LetterFeedback.WRONG_LETTER)
    return feedback

现在创建一个 `next_turn` 函数，把某次猜测得到的反馈并入发给 LLM 的提示词中：

In [12]:
def next_turn(
    past_guesses: List[GuessWithFeedback], 
    secret_word: str, 
    adapter_id = ""
):
    prompt = render_prompt(past_guesses)
    completion = generate_stream(prompt, adapter_id)
    match = re.search(
        r"<guess>\s*(.*?)\s*</guess>", completion, re.DOTALL
    )
    if not match:
        raise RuntimeError("invalid guess")
    
    guess = match.group(1).upper()
    feedback = get_feedback(guess, secret_word)
    past_guesses.append(GuessWithFeedback(guess, feedback))
    print("\n\n")
    print(("-" * 100) + "\n")
    for past_guess in past_guesses:
        print(past_guess)
    
    if guess == secret_word:
        print("🎉 SUCCESS 🎉")
    elif len(past_guesses) >= 6:
        print("❌ better luck next time... ❌")

试着用基础模型来玩：

In [13]:
secret_word = "BRICK"
past_guesses = []
adapter_id = ""

In [14]:
next_turn(past_guesses, secret_word, adapter_id)

 
- The word is 5 letters long.
- We have 6 tries to guess the word.
- We need to use valid 5-letter English words.
- We should try to use words that cover a wide range of letters to maximize our chances of getting feedback.
- A good starting word could be a common 5-letter word that includes a mix of vowels and consonants.
</think>
<guess> CRANE </guess>



----------------------------------------------------------------------------------------------------

CRANE → Feedback: C(-) R(✓) A(x) N(x) E(x)


现在试试微调后的模型：

In [15]:
secret_word = "BRICK"
past_guesses = []
adapter_id = "wordle-dlai/2"

In [16]:
next_turn(past_guesses, secret_word, adapter_id)

I need to make a first guess for a 5-letter word in Wordle. For the first guess, it's best to choose a word that:

1. Contains common letters
2. Has vowels
3. Has minimal repeated letters

Some good starting words include:
- ARISE (has A, I, E and common consonants R, S)
- TEARS (has E, A and common consonants T, R, S)
- STARE (same letters as TEARS, just rearranged)
- ADIEU (has many vowels: A, I, E, U)
- CRANE (has A, E and common consonants C, R, N)

I think I'll go with "STARE" as my first guess. It has:
- S: a very common letter
- T: another common letter
- A: the most common vowel
- R: another common letter
- E: the most common letter in English

This gives me a good mix of common consonants and vowels, which should provide useful information for my subsequent guesses.
For my first guess in Wordle, I'll choose a word with common letters that will give me good information.</think><guess>STARE</guess>



------------------------------------------------------------------------------

## 自己动手试试！

把上面的 secret word 换成其他单词，看看模型会如何响应。 